In [ ]:
# --- Multimodal chat (imagen + texto) en OCI Generative AI---

import os, json, base64, mimetypes
import oci


## Debemos actualizar los datos del compartment para la ejecución del código

In [ ]:
# === Configuración base ===
CONFIG_PROFILE = "DEFAULT"
config = oci.config.from_file(os.path.expanduser("~/.oci/config"), CONFIG_PROFILE)
config["region"] = "us-chicago-1"

##datos actualizar
COMPARTMENT_ID = "ocid1.compartment.oc1.."
ENDPOINT = "https://inference.generativeai.us-chicago-1.oci.oraclecloud.com"

MODEL_ID = "ocid1.generativeaimodel.oc1.us-chicago-1.amaaaaaask7dceyayjawvuonfkw2ua4bob4rlnnlhs522pafbglivtwlfzta"
assert ".oc1.us-chicago-1." in MODEL_ID, "El model_id debe ser de la región us-chicago-1"


In [ ]:


# === Utilidad: convertir imagen local a data URL base64 ===
def image_to_data_url(path: str) -> str:
    mime = mimetypes.guess_type(path)[0] or "image/jpeg"
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    return f"data:{mime};base64,{b64}"

# Ruta de tu imagen local
IMG_PATH = "demo-image.jpeg"
PROMPT = "Describe brevemente lo que ves en la imagen."

# Prepara contenidos multimodales
from oci.generative_ai_inference import GenerativeAiInferenceClient
from oci.generative_ai_inference.models import (
    ChatDetails,
    OnDemandServingMode,
    GenericChatRequest,
    BaseChatRequest,
    Message,
    TextContent,
    ImageContent,
    ImageUrl,
)

# Imagen como data URL (también puedes usar una URL pública HTTPS)
img_data_url = image_to_data_url(IMG_PATH)

image_url = ImageUrl(url=img_data_url)
image_content = ImageContent(image_url=image_url)

text_content = TextContent(text=PROMPT)

user_message = Message(role="USER", content=[text_content, image_content])

chat_request = GenericChatRequest(
    api_format=BaseChatRequest.API_FORMAT_GENERIC,
    messages=[user_message],
    max_tokens=400,
    temperature=0.4,
    top_p=0.9,
)

chat_details = ChatDetails(
    serving_mode=OnDemandServingMode(model_id=MODEL_ID),
    chat_request=chat_request,
    compartment_id=COMPARTMENT_ID,
)

client = GenerativeAiInferenceClient(config=config, service_endpoint=ENDPOINT)
resp = client.chat(chat_details)

# Extrae y muestra el texto de la respuesta
choices = resp.data.chat_response.choices
if choices:
    parts = choices[0].message.content
    texto = "\n".join(getattr(p, "text", "") for p in parts if hasattr(p, "text"))
    print("\n=== RESPUESTA ===\n", texto.strip())
else:
    print("No se generó respuesta.")